# Temporal Interference (TI) simulations with SimNIBS — a hands-on introduction

This notebook takes you from zero to a full TI electrode optimization, using only tools that are built into **SimNIBS 4.6**:

| Step | What you do | Main SimNIBS tool |
|---|---|---|
| 0 | Install SimNIBS, open this notebook, learn the bits of Python you need | `simnibs_jupyter` |
| 1 | Build a head model from MRI | `charm` |
| 2 | Look inside the head model | `read_msh`, `ElementTags` |
| 3 | Run your first tDCS simulation | `sim_struct.SESSION`, `run_simnibs` |
| 4 | Analyse the tDCS result | `crop_mesh`, field arrays |
| 5 | Create region masks (target + regions to avoid) | `RegionOfInterest`, `warp_volume` |
| 6 | From two tDCS fields to TI | `TI_utils.get_maxTI` |
| 7 | Leadfields: search thousands of cap montages | `sim_struct.TDCSLEADFIELD`, `TI_utils.get_field` |
| 8 | Let SimNIBS optimize the TI electrode positions freely | `opt_struct.TesFlexOptimization` |

**Running example:** the SimNIBS example subject *ernie*; the target is the **right hippocampus**, a deep structure. Regions we want to avoid are the **motor + premotor cortex**.

**How to use it:** run the cells from top to bottom (`Shift + Enter`). Cells marked ⏳ take a long time; they are protected by a `RUN_...` switch so they don't start by accident. Each step ends with a short **Try it** box.

---
## Step 0 — Setup

### 0.1 Install SimNIBS (once)

SimNIBS comes with its **own Python installation**, so you do *not* need to install Python, Anaconda or anything else first.

1. Go to <https://simnibs.github.io/simnibs/build/html/installation/simnibs_installer.html> and download the **SimNIBS 4.6 installer** for your system.
2. Install it:
   - **Windows:** run the `.exe`. Keep the default options. If Windows SmartScreen warns you, click *More info → Run anyway*.
   - **macOS:** open the `.pkg` and follow the steps. On the first launch macOS may block it: go to *System Settings → Privacy & Security* and click *Open anyway*.
   - **Linux:** extract the archive and run `./install` from a terminal inside the extracted folder.
3. Close and re-open your terminal, then check the installation:
   ```
   simnibs_python -c "import simnibs; print(simnibs.__version__)"
   ```
   It should print `4.6.x`.

Where is a "terminal"? **Windows:** press the Start key, type *cmd* and open *Command Prompt*. **macOS:** *Terminal* (in Applications → Utilities). **Linux:** your usual terminal.

### 0.2 What is a "Python environment"?

A Python **environment** is a folder that contains one Python interpreter plus a set of installed packages (numpy, nibabel, …). Keeping projects in separate environments avoids version conflicts.

The SimNIBS installer has already **created an environment for you** (a folder called `simnibs_env` inside the SimNIBS installation) and put three commands on your system:

| Command | What it does |
|---|---|
| `simnibs_python` | Runs Python *inside* the SimNIBS environment |
| `simnibs_jupyter` | Starts JupyterLab *inside* the SimNIBS environment |
| `charm`, `simnibs`, `gmsh`, … | SimNIBS command-line tools |

If you ever need an extra package, install it **into this environment** with:
```
simnibs_python -m pip install <package-name>
```
(Everything this notebook needs is already included.)

### 0.3 Download the example data

Download the **SimNIBS 4 example dataset** (`simnibs4_examples.zip`) from <https://simnibs.github.io/simnibs/build/html/dataset.html> and unzip it somewhere with a few GB of free space. You will use the raw MRI images in its `org/` folder (`ernie_T1.nii.gz`, `ernie_T2.nii.gz`).

You also need the **Brainnetome atlas** volume `BN_Atlas_246_1mm.nii.gz`. Download it from <https://atlas.brainnetome.org/download.html>, or ask the lab for a copy.

### 0.4 Open this notebook

In a terminal, go to the folder that contains this notebook and start JupyterLab:
```
cd path/to/TI_onboarding
simnibs_jupyter
```
A browser tab opens. Double-click this notebook, then check that the kernel in the top right says *Python 3*.

### 0.5 The bits of Python you need

You do not need to be a Python expert. These are the patterns used in this notebook. Run the cell below and compare its output with the code.

* `x = 5` stores a value in a **variable**.
* `[1, 2, 3]` is a **list**, and `{"a": 1}` is a **dictionary** (it looks values up by name).
* `import numpy as np` loads a **package** (a library of functions) under a short name.
* `np.array(...)` is a **numpy array**: a table of numbers you can do maths on all at once. A head model has millions of numbers, so we always work with arrays rather than loops.
* `values[mask]` keeps only the entries where `mask` is `True`. This **boolean mask** is the single most important trick in this notebook: "the field *in the hippocampus*" is literally `field[hippocampus_mask]`.
* `def f(x): ...` defines a **function**, a reusable piece of code.
* `f"{x:.2f}"` is an **f-string**: text with a value inserted (here with 2 decimals).
* `Path("a") / "b"` builds a file path that works on every operating system.

In [ ]:
import numpy as np
from pathlib import Path

field = np.array([0.10, 0.25, 0.05, 0.40])         # e.g. field strength in 4 mesh elements (V/m)
in_target = np.array([False, True, False, True])    # which elements belong to the target

print("all values     :", field)
print("target values  :", field[in_target])         # boolean mask
print(f"target mean    : {field[in_target].mean():.3f} V/m")
print("folder example :", Path("work") / "tdcs_simulation")

def double(x):
    # a tiny function
    return 2 * x
print("double(0.2)    :", double(0.2))

### 0.6 Imports and settings

**Edit the paths in the next cell**, then run it.

* `DATA_DIR`: the unzipped `simnibs4_examples` folder.
* `BNA_PATH`: the Brainnetome atlas file.
* `WORK_DIR`: where all results are written. It is created next to this notebook by default.

`OPEN_GMSH = True` opens a Gmsh 3D viewer window for some results. Close that window before you continue; the notebook waits until you do.

In [ ]:
import os, sys, gc, glob, copy, subprocess
from pathlib import Path

import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

import simnibs
from simnibs import sim_struct, run_simnibs, mesh_io, ElementTags, opt_struct, RegionOfInterest
from simnibs.utils import TI_utils as TI
from simnibs.utils.transformations import warp_volume
from simnibs.mesh_tools.gmsh_view import open_in_gmsh

# ---------------- EDIT THESE ----------------
DATA_DIR = Path(r"C:\path\to\simnibs4_examples")      # unzipped example dataset (contains org/)
BNA_PATH = Path(r"C:\path\to\BN_Atlas_246_1mm.nii.gz")  # Brainnetome atlas
WORK_DIR = Path("work").resolve()                      # all outputs go here
OPEN_GMSH = True                                        # open 3D viewer windows?
# --------------------------------------------

SUBJECT = "ernie"
T1 = DATA_DIR / "org" / "ernie_T1.nii.gz"
T2 = DATA_DIR / "org" / "ernie_T2.nii.gz"
M2M = WORK_DIR / f"m2m_{SUBJECT}"        # the head model folder created by charm (Step 1)
MASK_DIR = WORK_DIR / "masks"

WORK_DIR.mkdir(exist_ok=True)
MASK_DIR.mkdir(exist_ok=True)

print("SimNIBS version:", simnibs.__version__)
for p in [T1, T2, BNA_PATH]:
    print(("OK      " if p.exists() else "MISSING ") + str(p))

### 0.7 Helper functions

These small functions are used throughout the notebook. Run the cell once. You don't need to understand every line now; each function is explained where it is first used.

* `load_tetra(fn)` reads a mesh file and keeps only the **tetrahedra** (the 3D volume elements), dropping the surface triangles.
* `roi_elements(mesh, roi)` finds which elements of `mesh` belong to a SimNIBS `RegionOfInterest`. It returns their positions (indices) in the mesh. `elements_at(mesh, centres)` does the same from a stored list of points.
* `summary(mesh, values, idx)` computes statistics of `values` inside a region, **weighted by element volume** (see Step 4 for why).
* `plot_slice(...)` gives a quick 2D picture of a field on a thin slab through the head.

> **Memory warning.** A head mesh takes several GB of RAM once loaded, and simulation results are even bigger. A laptop can run out of memory quickly. This notebook therefore (1) loads the head mesh **once** and shares it, (2) deletes big objects when a step is done (`del x` followed by `gc.collect()`), and (3) stores regions as small lists of points rather than as full mesh objects. If your computer becomes slow, restart the kernel (*Kernel → Restart*) and re-run from the top; finished simulations are not recomputed.

In [ ]:
def load_tetra(fn):
    "Read a .msh file and keep only tetrahedra (3D elements)."
    return mesh_io.read_msh(str(fn)).crop_mesh(elm_type=4)


def elements_at(mesh, centres):
    "0-based indices of the tetrahedra in `mesh` that contain the given points."
    idx = mesh.find_tetrahedron_with_points(centres, compute_baricentric=False)
    return np.unique(idx[idx > 0] - 1)          # SimNIBS counts from 1, numpy from 0


def roi_elements(mesh, roi):
    "0-based indices of the tetrahedra in `mesh` that contain the ROI's element centres."
    return elements_at(mesh, roi.get_nodes("elm_center"))


def weighted_percentile(values, weights, q):
    order = np.argsort(values)
    cum = np.cumsum(weights[order]) / weights.sum()
    return values[order][np.searchsorted(cum, q / 100)]


def summary(mesh, values, idx):
    "Volume-weighted statistics of `values` (one value per element) inside region `idx`."
    vols = mesh.elements_volumes_and_areas().value[idx]
    v = values[idx]
    return {
        "volume_cm3": vols.sum() / 1000,
        "mean": np.average(v, weights=vols),
        "median": weighted_percentile(v, vols, 50),
        "p99": weighted_percentile(v, vols, 99),
    }


def plot_slice(mesh, values, axis="y", coord=0.0, thickness=1.5, ax=None,
               vmax=None, title="", cmap="viridis", label="V/m"):
    "Scatter-plot element values within +/- thickness mm of a plane (axis = 'x', 'y' or 'z')."
    a = "xyz".index(axis)
    c = mesh.elements_baricenters().value
    sel = np.abs(c[:, a] - coord) < thickness
    h, v = [i for i in range(3) if i != a]
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    sc = ax.scatter(c[sel, h], c[sel, v], c=values[sel], s=1, cmap=cmap, vmin=0, vmax=vmax)
    ax.set_aspect("equal"); ax.set_title(title)
    ax.set_xlabel("xyz"[h] + " (mm)"); ax.set_ylabel("xyz"[v] + " (mm)")
    plt.colorbar(sc, ax=ax, label=label, shrink=0.8)
    return ax

print("helpers loaded")

---
## Step 1 — Build a head model with `charm` ⏳ (1–2 hours)

A simulation needs a **head model**: a 3D mesh in which every small tetrahedron is labelled with a tissue type (scalp, bone, CSF, grey matter, …). SimNIBS builds it automatically from a T1-weighted MRI (and optionally a T2) with **`charm`**.

charm writes a folder called `m2m_<subject>` that contains everything later steps need:

| In `m2m_ernie/` | What it is |
|---|---|
| `ernie.msh` | the tetrahedral head mesh |
| `segmentation/labeling.nii.gz` | a detailed label volume (~50 structures, incl. hippocampus); used in Step 5 |
| `toMNI/` | the warp between this head and the MNI template; used to bring atlases into the subject |
| `surfaces/` | the cortical surfaces (incl. the *central* surface halfway through grey matter) |
| `eeg_positions/` | EEG 10-10 electrode positions on this scalp (e.g. `C3`, `F5`) |
| `charm_report.html` | **open this** to check the segmentation visually |

**Recommended: run it from a terminal** so you can watch the progress:
```
cd <WORK_DIR shown above>
charm ernie <DATA_DIR>/org/ernie_T1.nii.gz <DATA_DIR>/org/ernie_T2.nii.gz
```
Alternatively, set `RUN_CHARM = True` below and run the cell. The notebook will be busy for 1–2 h, and the progress log goes to the terminal that started Jupyter.

> Note: the example dataset also ships a ready-made `m2m_ernie`, but **without** the `segmentation/` folder that we need for the hippocampus label. That's why we build our own. Building it is also the first thing you'll do for every real subject.

In [ ]:
RUN_CHARM = False   # ⏳ set to True to run charm from the notebook

if (M2M / f"{SUBJECT}.msh").exists():
    print("Head model already exists:", M2M)
elif RUN_CHARM:
    cmd = [sys.executable, "-m", "simnibs.cli.charm", SUBJECT, str(T1), str(T2)]
    print("Running:", " ".join(cmd))
    subprocess.run(cmd, cwd=WORK_DIR, check=True)
else:
    print("No head model yet: run charm (terminal or RUN_CHARM = True).")

> **Try it:** open `m2m_ernie/charm_report.html` in your browser. Scroll through the overlays: do the tissue boundaries (skull, CSF, grey matter) follow the MRI?

---
## Step 2 — Look inside the head model

A mesh has:
* **nodes**: 3D points (x, y, z in mm, in the subject's own MRI space);
* **elements**: tetrahedra (volume) and triangles (surfaces between tissues), each built from 3–4 nodes;
* a **tag** per element saying which tissue it belongs to.

SimNIBS gives the tags names through `ElementTags`. Tags 1–10 are volumes and 1001–1010 are the surfaces between them. Electrodes and gel (100/500) are added later by the simulation.

`simnibs.read_msh(path)` loads a mesh. `mesh.elm.tag1` is the array of tissue tags, one per element.

In [ ]:
head = load_tetra(M2M / f"{SUBJECT}.msh")
print(f"{head.nodes.nr:,} nodes, {head.elm.nr:,} tetrahedra")

vols = head.elements_volumes_and_areas().value      # volume of each tetrahedron (mm^3)
rows = []
for tag in np.unique(head.elm.tag1):
    sel = head.elm.tag1 == tag
    rows.append({"tag": tag, "tissue": ElementTags(tag).name,
                 "n_elements": sel.sum(), "volume_cm3": vols[sel].sum() / 1000})
pd.DataFrame(rows)

Each tissue gets an electrical **conductivity** (S/m). These are SimNIBS's standard values. Notice how poorly the skull conducts compared with the scalp and CSF. This is why most of the current applied on the scalp never reaches the brain.

In [ ]:
from simnibs.utils import cond_utils
conds = cond_utils.standard_cond()
pd.DataFrame([{"tag": i + 1, "tissue": c.name, "conductivity_S_per_m": c.value}
              for i, c in enumerate(conds[:10])])

> **Try it:** the scalp is the outermost layer. How many cm³ of scalp does ernie have, compared with grey matter? Find the tag of the *CSF* in the table and compute its share of the total head volume.

---
## Step 3 — Your first tDCS simulation

Transcranial direct current stimulation (tDCS) means injecting a constant current through scalp electrodes. To simulate it, SimNIBS solves the electrostatic equations on the mesh and gives you the **electric field E** (V/m) in every element.

A simulation is described by nested settings objects:

* `sim_struct.SESSION()`: the whole job (which head, which output folder).
* `S.add_tdcslist()`: one stimulation condition. It holds the `currents` per channel, in **amperes** (0.001 A = 1 mA). The currents must sum to zero: what goes in must come out.
* `tdcs.add_electrode()`: one electrode. `channelnr` links it to a current, `centre` is an EEG 10-10 position name, and `shape`/`dimensions`/`thickness` describe the pad (mm).
* `run_simnibs(S)`: runs it, which takes a few minutes.

This is the standard SimNIBS example: an anode over **C3** (left motor cortex) and a cathode over **AF4** (right forehead), 1 mA.

In [ ]:
TDCS_DIR = WORK_DIR / "tdcs_simulation"
fn_tdcs = TDCS_DIR / f"{SUBJECT}_TDCS_1_scalar.msh"

S = sim_struct.SESSION()
S.subpath = str(M2M)          # which head model
S.pathfem = str(TDCS_DIR)     # where to write results
S.map_to_surf = True          # also map results onto the cortical (central) surface
S.open_in_gmsh = OPEN_GMSH

tdcs = S.add_tdcslist()
tdcs.currents = [0.001, -0.001]   # channel 1: +1 mA, channel 2: -1 mA

anode = tdcs.add_electrode()
anode.channelnr = 1
anode.centre = "C3"
anode.shape = "rect"
anode.dimensions = [50, 50]       # mm
anode.thickness = 4               # mm

cathode = tdcs.add_electrode()
cathode.channelnr = 2
cathode.centre = "AF4"
cathode.shape = "rect"
cathode.dimensions = [50, 70]
cathode.thickness = 4

if fn_tdcs.exists():
    print("Already simulated:", fn_tdcs)
else:
    run_simnibs(S)            # a few minutes

The output folder contains:
* `ernie_TDCS_1_scalar.msh`: the head mesh plus electrodes, with the fields `E` (vector), `magnE` (its strength), `J`/`magnJ` (current density) and `v` (potential).
* `subject_overlays/`: the same fields on the central cortical surface (because `map_to_surf = True`).
* `.geo` / `.opt` files that tell Gmsh how to display the result.

> **Try it:** copy the cell, move the anode to `C4` (right motor cortex), change the output folder to `tdcs_C4`, and run it. Then use Step 4 on that result as well.

---
## Step 4 — Analyse the tDCS result

Typical questions are: *how strong is the field in the brain?* and *how strong is it in my region of interest?*

1. Load the result and keep only **grey matter (GM)**. `mesh.crop_mesh(tags=...)` returns a smaller mesh with only those elements.
2. Take `magnE`, the field strength in each element.
3. Summarise it. The elements have **different sizes**: small ones are packed where the anatomy is detailed. A plain average over elements would over-count those regions, so we **weight every element by its volume**.
4. Report robust numbers. The single maximum is often a mesh artefact (one tiny element), so we report the **99th percentile (p99)**, a "peak" that ignores the top 1 % of the volume.

In [ ]:
res = load_tetra(fn_tdcs)
gm_idx = np.where(res.elm.tag1 == ElementTags.GM)[0]
magnE = res.field["magnE"].value

s = summary(res, magnE, gm_idx)
print(f"Grey matter: volume {s['volume_cm3']:.0f} cm3 | mean {s['mean']:.3f} | median {s['median']:.3f} "
      f"| p99 {s['p99']:.3f} | max {magnE[gm_idx].max():.3f}  (V/m)")

vols = res.elements_volumes_and_areas().value
plt.figure(figsize=(6, 3))
plt.hist(magnE[gm_idx], bins=100, weights=vols[gm_idx] / 1000)
plt.axvline(s["p99"], color="k", ls="--", label="p99")
plt.xlabel("|E| (V/m)"); plt.ylabel("GM volume (cm³)"); plt.legend(); plt.title("tDCS field in grey matter")
plt.show()

**Field in a region of interest.** Here we define left M1 as a 10 mm sphere around the MNI coordinate (−37, −21, 58), the value used in the SimNIBS example, and keep only grey matter.

`RegionOfInterest` is SimNIBS's general tool for defining regions. You'll meet it again in Step 5.
* `simnibs.mni2subject_coords` converts the MNI coordinate into ernie's own space (mm).
* `roi.mesh = head` hands it the head mesh we already loaded. (Alternatively, `roi.subpath = M2M` makes it load its own copy from disk. That's simpler, but costs another few GB of RAM per region.)

In [ ]:
m1_subject = simnibs.mni2subject_coords([-37, -21, 58], str(M2M))

roi_m1 = RegionOfInterest()
roi_m1.mesh = head
roi_m1.method = "volume"
roi_m1.tissues = [ElementTags.GM]
roi_m1.roi_sphere_center_space = "subject"
roi_m1.roi_sphere_center = list(m1_subject)
roi_m1.roi_sphere_radius = 10

m1_idx = roi_elements(res, roi_m1)
s = summary(res, magnE, m1_idx)
print(f"Left M1 sphere: {s['volume_cm3']:.2f} cm3 | mean {s['mean']:.3f} | p99 {s['p99']:.3f} V/m")

plot_slice(res, magnE, axis="y", coord=m1_subject[1], title="|E| on a coronal slab through left M1")
plt.show()

To explore the field in 3D, open the result in **Gmsh** (it is bundled with SimNIBS). In Gmsh: the left panel lists the views. Hide/show tissues with *Tools → Visibility*, and change the colour range with *right-click on a view → Options*.

In [ ]:
if OPEN_GMSH:
    open_in_gmsh(str(fn_tdcs))

# free the memory used by the tDCS result before moving on
del res, magnE
gc.collect()

> **Try it:** (1) Compute the same M1 numbers for the `C4` simulation from Step 3 (load it with `load_tetra`). (2) Move the sphere to the right M1: convert MNI (37, −21, 58) with `mni2subject_coords` and use that as `roi_sphere_center`. Which montage stimulates which hemisphere more? (3) Compare `p99` and `max` for GM: why are they so different?

---
## Step 5 — Create masks

For TI we need two kinds of region:
* the **target** (ROI): the right hippocampus;
* regions to **avoid** (non-ROI): the motor + premotor cortex.

There are several ways to define "the right hippocampus". We build three and compare them:

| Method | Source | Pros / cons |
|---|---|---|
| **A. MNI sphere** | a coordinate + radius | simplest, but a sphere is not a hippocampus shape |
| **B. charm label** | `segmentation/labeling.nii.gz`, label 53 = *Right-Hippocampus* | subject-specific segmentation, no warp needed |
| **C. Atlas warped from MNI** | Brainnetome atlas, labels 216 (rostral) + 218 (caudal hippocampus, right) | any MNI atlas works this way; depends on the warp quality |

All three end up as a `RegionOfInterest` with `method = "volume"`, so we can compare them element by element. Here we include WM, GM and CSF elements, to see which tissues each mask actually covers.

### 5A. Sphere around an MNI coordinate

The centre (26, −21, −15) is the centre of mass of the Brainnetome right hippocampus (labels 216 + 218) in MNI space, with a 10 mm radius. The real hippocampus is ~4 cm long and 1–2 cm thick, so this is only a rough stand-in.

In [ ]:
HIPPO_MNI = [26, -21, -15]
BRAIN = [ElementTags.WM, ElementTags.GM, ElementTags.CSF]

hippo_subject = simnibs.mni2subject_coords(HIPPO_MNI, str(M2M))
print("Sphere centre in ernie's space (mm):", np.round(hippo_subject, 1))

roi_sphere = RegionOfInterest()
roi_sphere.mesh = head                      # share the already-loaded head mesh
roi_sphere.method = "volume"
roi_sphere.tissues = BRAIN
roi_sphere.roi_sphere_center_space = "subject"
roi_sphere.roi_sphere_center = list(hippo_subject)
roi_sphere.roi_sphere_radius = 10

### 5B. charm's own label

`labeling.nii.gz` is a 3D image in which every voxel holds a label number. Its look-up table (`labeling_LUT.txt`) says that 53 = *Right-Hippocampus*. We make a **binary mask image** (1 inside, 0 outside), save it, and give it to `RegionOfInterest` with `mask_space = "subject"`, meaning the image is already in ernie's space.

`nib.load` reads a NIfTI image. `np.asarray(img.dataobj)` gives the voxel values as an array, and `img.affine` is the 4×4 matrix that converts voxel indices into mm coordinates.

In [ ]:
lab_img = nib.load(M2M / "segmentation" / "labeling.nii.gz")
lab = np.squeeze(np.asarray(lab_img.dataobj))
with open(M2M / "segmentation" / "labeling_LUT.txt") as f:
    print([line.strip() for line in f if line.split()[:1] == ["53"]])

hippo_charm = (lab == 53).astype(np.uint8)
fn_hippo_charm = MASK_DIR / "hippocampus_R_charm.nii.gz"
nib.save(nib.Nifti1Image(hippo_charm, lab_img.affine), fn_hippo_charm)
print("voxels:", hippo_charm.sum(), "->", fn_hippo_charm)

roi_charm = RegionOfInterest()
roi_charm.mesh = head
roi_charm.method = "volume"
roi_charm.tissues = BRAIN
roi_charm.mask_path = str(fn_hippo_charm)
roi_charm.mask_space = "subject"
roi_charm.mask_value = 1

### 5C. Atlas in MNI space, warped into the subject

The Brainnetome atlas is defined on the MNI template. `warp_volume(..., transformation_direction="mni2subject")` uses the non-linear warp that charm computed (`toMNI/`) to resample it into ernie's space. `order=0` means **nearest-neighbour**, which is essential for label images: we want label 216 or 218, never an average such as 217.

(Shortcut: `RegionOfInterest` can also take the MNI atlas directly with `mask_space = "mni"` and warp it internally. We do it explicitly here so you can open the warped image and check it.)

In [ ]:
fn_bna_subject = MASK_DIR / "BNA_in_ernie.nii.gz"
if not fn_bna_subject.exists():
    warp_volume(str(BNA_PATH), str(M2M), str(fn_bna_subject),
                transformation_direction="mni2subject", order=0)

bna_img = nib.load(fn_bna_subject)
bna = np.rint(np.squeeze(np.asarray(bna_img.dataobj))).astype(int)
hippo_bna = np.isin(bna, [216, 218]).astype(np.uint8)
fn_hippo_bna = MASK_DIR / "hippocampus_R_BNA216_218.nii.gz"
nib.save(nib.Nifti1Image(hippo_bna, bna_img.affine), fn_hippo_bna)
print("voxels:", hippo_bna.sum(), "->", fn_hippo_bna)

roi_bna = RegionOfInterest()
roi_bna.mesh = head
roi_bna.method = "volume"
roi_bna.tissues = BRAIN
roi_bna.mask_path = str(fn_hippo_bna)
roi_bna.mask_space = "subject"
roi_bna.mask_value = 1

### 5D. Compare the three hippocampus masks

For each mask, the table shows its volume in each tissue. Below it, the **Dice overlap** between each pair of masks: 2·|A∩B| / (|A|+|B|), which is 1 for identical masks and 0 for no overlap. (Here it's computed on element counts.)

What to expect: charm puts the hippocampus mostly in **grey matter**. The sphere also grabs white matter, CSF (the ventricle next to the hippocampus) and neighbouring structures.

In [ ]:
hippo_masks = {"A_sphere": roi_sphere, "B_charm53": roi_charm, "C_BNA216_218": roi_bna}
hippo_idx = {name: roi_elements(head, roi) for name, roi in hippo_masks.items()}

vols = head.elements_volumes_and_areas().value
rows = []
for name, idx in hippo_idx.items():
    row = {"mask": name, "total_cm3": vols[idx].sum() / 1000}
    for t in BRAIN:
        row[f"{t.name}_cm3"] = vols[idx][head.elm.tag1[idx] == t].sum() / 1000
    rows.append(row)
display(pd.DataFrame(rows).round(2))

names = list(hippo_idx)
dice = pd.DataFrame(index=names, columns=names, dtype=float)
for a in names:
    for b in names:
        A, B = set(hippo_idx[a]), set(hippo_idx[b])
        dice.loc[a, b] = 2 * len(A & B) / (len(A) + len(B))
print("Dice overlap:")
dice.round(2)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (name, idx) in zip(axes, hippo_idx.items()):
    m = np.zeros(head.elm.nr); m[idx] = 1
    brain_tags = np.isin(head.elm.tag1, [1, 2, 3])
    plot_slice(head, np.where(brain_tags, 0.3 + 0.7 * m, np.nan), axis="y",
               coord=hippo_subject[1], ax=ax, vmax=1, title=name, cmap="magma", label="")
    ax.set_xlim(hippo_subject[0] - 40, hippo_subject[0] + 40)
    ax.set_ylim(hippo_subject[2] - 40, hippo_subject[2] + 40)
plt.suptitle("Coronal slab through the hippocampus (bright = mask)")
plt.show()

To check the masks in 3D, save them as fields on the brain mesh and open it in Gmsh.

In [ ]:
brain_mesh = head.crop_mesh(tags=BRAIN)
for name, roi in hippo_masks.items():
    m = np.zeros(brain_mesh.elm.nr); m[roi_elements(brain_mesh, roi)] = 1
    brain_mesh.add_element_field(m, name)
fn_masks_vis = MASK_DIR / "hippocampus_masks.msh"
mesh_io.write_msh(brain_mesh, str(fn_masks_vis))
print("wrote", fn_masks_vis)
del brain_mesh; gc.collect()            # free memory
if OPEN_GMSH:
    open_in_gmsh(str(fn_masks_vis))

### 5E. Regions to avoid: motor + premotor cortex from a built-in SimNIBS atlas

SimNIBS ships cortical atlases defined on the *fsaverage* template surface: `DK40` (Desikan-Killiany), `HCP_MMP1` and `a2009s`. SimNIBS maps them onto each subject's own cortical surface.

We use DK40's **precentral** gyrus (primary motor cortex, plus part of premotor) and **caudal middle frontal** gyrus (dorsal premotor), in **both hemispheres**. A `.annot` file stores one number per region, so we first look these numbers up by name.

`method = "volume_from_surface"` turns the selected patch of the central cortical surface into a **volume**: it keeps the grey-matter tetrahedra within `surface_inclusion_radius = 5` mm of it.

How the four atlas pieces are combined (`mask_operator`): the region starts as the whole cortical surface. The first operator, `"intersection"`, cuts it down to the left precentral gyrus, and the next three, `"union"`, add the other parts.

This region needs `subpath`, because it must read the cortical surfaces from the m2m folder. That also makes it load its own copy of the head mesh. So once we have its element centres (a small list of points), we delete the object to free the memory.

In [ ]:
ATLAS_DIR = Path(simnibs.SIMNIBSDIR) / "resources" / "templates" / "fsaverage_atlases"

def annot_index(hemi, region, atlas="DK40"):
    _, _, names = nib.freesurfer.io.read_annot(str(ATLAS_DIR / f"{hemi}.aparc_{atlas}.annot"))
    return [n.decode() for n in names].index(region)

MOTOR_REGIONS = ["precentral", "caudalmiddlefrontal"]
paths, spaces, values = [], [], []
for hemi in ["lh", "rh"]:
    for region in MOTOR_REGIONS:
        paths.append(str(ATLAS_DIR / f"{hemi}.aparc_DK40.annot"))
        spaces.append(f"fs_avg_{hemi}")
        values.append(annot_index(hemi, region))
print(list(zip(spaces, values)))

roi_motor = RegionOfInterest()
roi_motor.subpath = str(M2M)
roi_motor.method = "volume_from_surface"
roi_motor.surface_type = "central"
roi_motor.mask_path = paths
roi_motor.mask_space = spaces
roi_motor.mask_value = values
roi_motor.mask_operator = ["intersection"] + ["union"] * (len(paths) - 1)
roi_motor.surface_inclusion_radius = 5          # mm; tissues defaults to grey matter

motor_idx = roi_elements(head, roi_motor)
x = head.elements_baricenters().value[motor_idx, 0]
print(f"motor/premotor: {vols[motor_idx].sum()/1000:.1f} cm3 "
      f"(left {vols[motor_idx][x < 0].sum()/1000:.1f}, right {vols[motor_idx][x > 0].sum()/1000:.1f})")

roi_motor.write_visualization(str(MASK_DIR), "motor_premotor_DK40")
motor_centres = roi_motor.get_nodes("elm_center")      # keep only the points ...
del roi_motor; gc.collect()                            # ... and free the mesh copy
if OPEN_GMSH:
    open_in_gmsh(str(MASK_DIR / "motor_premotor_DK40.msh"))

From here on, the **target** is mask B (charm label 53) restricted to **grey matter**, which is the default of `RegionOfInterest`. Over 95 % of it is GM anyway.

`REGIONS` stores each region as its list of element-centre points. Later we can find the matching elements in *any* result mesh with `elements_at(mesh, points)`.

> **Try it:** (1) Change the sphere radius to 5 and 15 mm and re-run the comparison: how do the volume and Dice change? (2) Use only BNA label 216 (rostral hippocampus). (3) Swap DK40 for `HCP_MMP1` and use areas `4`, `6a`, `6d`, `6v`, `6r` for the motor/premotor mask.

In [ ]:
roi_hippo = RegionOfInterest()
roi_hippo.mesh = head
roi_hippo.method = "volume"
roi_hippo.mask_path = str(fn_hippo_charm)
roi_hippo.mask_space = "subject"
roi_hippo.mask_value = 1                     # tissues defaults to grey matter

REGIONS = {"hippocampus_R (target)": roi_hippo.get_nodes("elm_center"),
           "motor+premotor (avoid)": motor_centres}
for name, pts in REGIONS.items():
    print(f"{name}: {len(pts):,} elements")

---
## Step 6 — From tDCS to Temporal Interference

**The idea.** Two electrode pairs each drive a sinusoidal current at a high frequency, for example pair 1 at 2000 Hz and pair 2 at 2010 Hz. Neurons barely respond to kHz fields. Where the two fields overlap, though, they add up and the sum **beats** at the difference frequency (10 Hz): its amplitude grows and shrinks 10 times per second. This slow amplitude change, the **envelope**, is thought to drive neurons. The envelope can be strong deep in the brain even where each single field is weaker than at the cortex.

**How SimNIBS computes it.** The fields are linear in the current, so we simply run **one tDCS simulation per pair** (Step 3, ×2). Then we combine the two field vectors **E1** and **E2** in each element with `TI_utils.get_maxTI(E1, E2)`. It returns the **maximal envelope amplitude** over all directions (Grossman et al., 2017). Two rules of thumb:
* if E1 and E2 point the same way, TI = 2 × the weaker field;
* if they are perpendicular, TI is smaller, and if one field is zero, TI is zero. TI needs **both** fields.

Check this on toy vectors (in V/m):

In [ ]:
E1 = np.array([[1.0, 0, 0], [1.0, 0, 0], [1.0, 0, 0], [1.0, 0, 0]])
E2 = np.array([[1.0, 0, 0], [0.3, 0, 0], [0, 1.0, 0], [0, 0, 0.0]])
for e1, e2, ti in zip(E1, E2, TI.get_maxTI(E1, E2)):
    print(f"E1={e1}  E2={e2}  ->  max TI = {ti:.2f} V/m")

### 6.1 Simulate the two pairs

This is the standard SimNIBS TI example: pair 1 is **F5–P5** (left) and pair 2 is **F6–P6** (right), round electrodes of 40 mm, 1 mA each. `copy.deepcopy` makes an independent copy of pair 1, which we then move to the right side.

In [ ]:
TI_DIR = WORK_DIR / "ti_simulation"

S = sim_struct.SESSION()
S.subpath = str(M2M)
S.pathfem = str(TI_DIR)
S.open_in_gmsh = False

pair1 = S.add_tdcslist()
pair1.currents = [0.001, -0.001]
for name in ["F5", "P5"]:
    el = pair1.add_electrode()
    el.channelnr = len(pair1.electrode)   # 1, then 2
    el.centre = name
    el.shape = "ellipse"
    el.dimensions = [40, 40]                # diameter (mm)
    el.thickness = 2

pair2 = S.add_tdcslist(copy.deepcopy(pair1))
pair2.electrode[0].centre = "F6"
pair2.electrode[1].centre = "P6"

fn_pair1 = TI_DIR / f"{SUBJECT}_TDCS_1_scalar.msh"
fn_pair2 = TI_DIR / f"{SUBJECT}_TDCS_2_scalar.msh"
if fn_pair2.exists():
    print("Already simulated:", TI_DIR)
else:
    run_simnibs(S)        # two simulations, a few minutes each

### 6.2 Compute the TI envelope

The two result meshes contain different electrodes. We keep only the head tissues (`load_tetra` + dropping electrode/gel tags), so both meshes have exactly the same elements. Then we combine the fields element by element.

In [ ]:
def compute_ti(fn1, fn2):
    "Load two tDCS results, keep head tetrahedra, return (mesh, |E1|, |E2|, TImax)."
    head_tags = np.arange(ElementTags.TH_START, ElementTags.SALINE_START - 1)
    m1 = load_tetra(fn1).crop_mesh(tags=head_tags)
    m2 = load_tetra(fn2).crop_mesh(tags=head_tags)
    assert m1.elm.nr == m2.elm.nr, "meshes differ"
    e1, e2 = m1.field["E"].value, m2.field["E"].value
    del m2; gc.collect()                       # only one mesh is needed from here on
    m1.elmdata = []                            # drop the fields we no longer need
    return m1, np.linalg.norm(e1, axis=1), np.linalg.norm(e2, axis=1), TI.get_maxTI(e1, e2)

ti_mesh, magE1, magE2, ti = compute_ti(fn_pair1, fn_pair2)

def region_table(mesh, fields):
    rows = []
    for rname, points in REGIONS.items():
        idx = elements_at(mesh, points)
        for fname, vals in fields.items():
            s = summary(mesh, vals, idx)
            rows.append({"region": rname, "field": fname, "mean": s["mean"], "p99": s["p99"]})
    return pd.DataFrame(rows).pivot(index="region", columns="field", values=["mean", "p99"]).round(3)

ti_table_step6 = region_table(ti_mesh, {"|E1|": magE1, "|E2|": magE2, "TI": ti})
ti_table_step6

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
vmax = np.percentile(np.r_[magE1, magE2], 99)
for ax, (name, vals) in zip(axes, {"|E1| (F5-P5)": magE1, "|E2| (F6-P6)": magE2, "TI max": ti}.items()):
    brain = np.isin(ti_mesh.elm.tag1, [1, 2])
    plot_slice(ti_mesh, np.where(brain, vals, np.nan), axis="y", coord=hippo_subject[1],
               ax=ax, vmax=vmax, title=name)
plt.suptitle("Coronal slab through the right hippocampus (WM + GM)")
plt.show()

Save the TI field for Gmsh, as in the SimNIBS example. The view shows TImax on the grey-matter surface.

In [ ]:
ti_mesh.elmdata = []
ti_mesh.add_element_field(magE1, "magnE - pair 1")
ti_mesh.add_element_field(magE2, "magnE - pair 2")
ti_mesh.add_element_field(ti, "TImax")
fn_ti = TI_DIR / "TI.msh"
mesh_io.write_msh(ti_mesh, str(fn_ti))
print("wrote", fn_ti)
if OPEN_GMSH:
    open_in_gmsh(str(fn_ti))

**Reading the table.** Compare TI in the hippocampus with TI in the motor areas. Also compare TI with the single-pair fields |E1| and |E2|: TI is high only where *both* fields are present.

> **Try it:** (1) Swap the electrodes of pair 2 (`P6`→`F6`, `F6`→`P6`). Predict first: does TI change? Then check. (2) Make pair 1 1.5 mA and pair 2 0.5 mA (same total of 2 mA): where does the TI maximum move? (3) Try `TI.get_dirTI(E1, E2, direction)` to get the envelope along a fixed direction, for example `[0, 0, 1]`.

---
## Step 7 — Leadfields: evaluate thousands of montages in minutes

In Step 6, every new montage costs two FEM simulations (minutes each). To compare *many* montages, we use **linearity**: the field of any electrode pair is a weighted sum of single-electrode fields.

1. **Compute a leadfield once** (`sim_struct.TDCSLEADFIELD`). SimNIBS places a small electrode at **every** EEG 10-10 position (76 on ernie's cap) and runs one simulation per electrode. Each simulation sends 1 mA from that electrode to a fixed **reference** electrode (here `Cz`). The result is the field E_k of every electrode k, stored in one `.hdf5` file.
2. **Any pair (A → B) is then simple arithmetic:** E = I × (E_A − E_B), because the reference cancels out. `TI.get_field([A, B, I], leadfield, idx)` does exactly this.
3. **TI for two pairs** = `TI.get_maxTI(E_pair1, E_pair2)`, as in Step 6.

The price: montages are limited to **cap positions** and to the **electrode size used in the leadfield**. All electrodes sit on the head at the same time, so they must be small. We use SimNIBS's default: round, 10 mm diameter, 4 mm thick.

Our settings:
* `interpolation = None` keeps the field in the **volume** elements, which is needed for a deep target like the hippocampus. The default would keep only the cortical surface.
* `tissues = [WM, GM]` stores only white and grey matter. That's enough for our regions and keeps the file much smaller.

### 7.1 Compute the leadfield ⏳ (roughly 30–60 min, several GB of RAM)

In [ ]:
RUN_LEADFIELD = False   # ⏳

# Step 6's mesh is no longer needed: free its memory
for name in ["ti_mesh", "magE1", "magE2", "ti"]:
    globals().pop(name, None)
gc.collect()

LF_DIR = WORK_DIR / "leadfield"
fn_lf = LF_DIR / f"{SUBJECT}_leadfield_EEG10-10_UI_Jurak_2007.hdf5"

lf = sim_struct.TDCSLEADFIELD()
lf.subpath = str(M2M)
lf.pathfem = str(LF_DIR)
lf.interpolation = None                         # volume leadfield (deep targets)
lf.tissues = [ElementTags.WM, ElementTags.GM]   # store WM + GM only
# electrode: SimNIBS default = round, 10 mm diameter, 4 mm thick

if fn_lf.exists():
    print("Leadfield already exists:", fn_lf)
elif RUN_LEADFIELD:
    run_simnibs(lf)
else:
    print("Set RUN_LEADFIELD = True to compute the leadfield.")

### 7.2 Load only what we need

The full leadfield is big: 75 electrodes × every WM/GM element × 3 field components. We don't need all of it, so we read just:
* the **mesh** stored inside the file (the WM + GM elements);
* for each electrode we use, its field **in our two regions only**.

`h5py` reads parts of an `.hdf5` file without loading the whole thing. `ds[i]` reads electrode *i*, and `[idx]` keeps our region's elements. `TI.load_leadfield(fn)` would load everything at once, which is fine on a cluster but heavy on a laptop.

In [ ]:
import h5py

POOL = ["Fp1", "Fp2", "F7", "F3", "Fz", "F4", "F8", "T7", "C3", "Cz",
        "C4", "T8", "P7", "P3", "Pz", "P4", "P8", "O1", "O2"]   # standard 10-20 positions
STEP6 = ["F5", "P5", "F6", "P6"]                               # for the comparison in 7.3
NEEDED = POOL + [e for e in STEP6 if e not in POOL]

lf_mesh = mesh_io.Msh.read_hdf5(str(fn_lf), "/mesh_leadfield/")
roi_idx = {name: elements_at(lf_mesh, pts) for name, pts in REGIONS.items()}
lf_vols = lf_mesh.elements_volumes_and_areas().value
roi_w = {name: lf_vols[idx] for name, idx in roi_idx.items()}     # element volumes = weights
all_idx = np.concatenate(list(roi_idx.values()))
n_elm = lf_mesh.elm.nr
del lf_mesh, lf_vols; gc.collect()

with h5py.File(fn_lf, "r") as f:
    ds = f["/mesh_leadfield/leadfields/tdcs_leadfield"]
    as_str = lambda x: x.decode() if isinstance(x, bytes) else str(x)
    ref = as_str(ds.attrs["reference_electrode"])
    names = [as_str(n) for n in ds.attrs["electrode_names"] if as_str(n) != ref]
    assert ds.shape[1] == n_elm, "leadfield and mesh sizes differ"
    print(f"leadfield: {ds.shape[0]} electrodes x {ds.shape[1]:,} elements x 3; reference = {ref}")
    sub_lf = np.stack([ds[names.index(e)][all_idx] for e in NEEDED if e != ref])

# index of each electrode in sub_lf; the reference has None (its field is zero by definition)
idx_sub = {e: i for i, e in enumerate([e for e in NEEDED if e != ref])}
idx_sub[ref] = None

# position of each region inside the concatenated `all_idx`
offsets = np.cumsum([0] + [len(v) for v in roi_idx.values()])
roi_slice = {name: slice(offsets[i], offsets[i + 1]) for i, name in enumerate(roi_idx)}
print("sub-leadfield shape:", sub_lf.shape, f"({sub_lf.nbytes / 1e6:.0f} MB)")

### 7.3 Rebuild the Step 6 montage from the leadfield

F5–P5 + F6–P6 at 1 mA, as in Step 6, but now built from the leadfield. The numbers will **not** be identical to Step 6. Step 6 used 40 mm electrodes and the leadfield uses 10 mm ones, and in the leadfield all the other cap electrodes are also sitting on the scalp. The point is to see the same overall picture, and how much electrode size matters.

In [ ]:
def wmean(values, w):
    return np.sum(values * w) / np.sum(w)

e1 = TI.get_field(["F5", "P5", 0.001], sub_lf, idx_sub)
e2 = TI.get_field(["F6", "P6", 0.001], sub_lf, idx_sub)
ti_lf = TI.get_maxTI(e1, e2)

rows = []
for name in roi_idx:
    s = roi_slice[name]
    rows.append({"region": name,
                 "leadfield (10 mm) mean TI": wmean(ti_lf[s], roi_w[name]),
                 "Step 6 (40 mm) mean TI": ti_table_step6.loc[name, ("mean", "TI")]})
pd.DataFrame(rows).set_index("region").round(3)

### 7.4 Exhaustive search over the 10-20 positions

Every way to choose **two disjoint pairs** from the 19 positions: 171 possible pairs, which gives **11,628 montages**. Each pair gets **2 mA**, as in Step 8. For every montage we compute TI in both regions and record:
* **intensity** = volume-weighted mean TI in the right hippocampus;
* **selectivity** = mean TI in the hippocampus ÷ mean TI in the motor + premotor cortex. Above 1 means the target gets more than the regions we want to avoid.

The polarity of a pair doesn't matter for `get_maxTI`, so A→B and B→A count once. This takes a few minutes.

In [ ]:
from itertools import combinations
import time

I = 0.002                                            # 2 mA per pair
HIP, MOT = "hippocampus_R (target)", "motor+premotor (avoid)"
pairs = list(combinations(POOL, 2))
montages = [(p, q) for i, p in enumerate(pairs) for q in pairs[i + 1:] if not set(p) & set(q)]
print(len(pairs), "pairs ->", len(montages), "montages")

pair_field = {p: TI.get_field([p[0], p[1], I], sub_lf, idx_sub) for p in pairs}   # cached per pair

t0, rows = time.time(), []
for k, (p, q) in enumerate(montages):
    t = TI.get_maxTI(pair_field[p], pair_field[q])
    hip = wmean(t[roi_slice[HIP]], roi_w[HIP])
    mot = wmean(t[roi_slice[MOT]], roi_w[MOT])
    rows.append({"pair 1": "-".join(p), "pair 2": "-".join(q),
                 "hippocampus mean TI": hip, "motor mean TI": mot, "selectivity": hip / mot})
    if k % 2000 == 0:
        print(f"{k:6d}/{len(montages)}  {time.time() - t0:5.0f} s")
search = pd.DataFrame(rows)
print(f"done in {time.time() - t0:.0f} s")

In [ ]:
print("Top 5 by INTENSITY")
display(search.nlargest(5, "hippocampus mean TI").round(3))
print("Top 5 by SELECTIVITY")
display(search.nlargest(5, "selectivity").round(3))

best_i = search["hippocampus mean TI"].idxmax()
best_s = search["selectivity"].idxmax()
plt.figure(figsize=(6, 5))
plt.scatter(search["motor mean TI"], search["hippocampus mean TI"], s=2, alpha=0.3, label="all montages")
plt.scatter(*search.loc[best_i, ["motor mean TI", "hippocampus mean TI"]], c="r", s=60, label="best intensity")
plt.scatter(*search.loc[best_s, ["motor mean TI", "hippocampus mean TI"]], c="k", marker="*", s=120, label="best selectivity")
lim = max(search["motor mean TI"].max(), search["hippocampus mean TI"].max())
plt.plot([0, lim], [0, lim], "k--", lw=0.8, label="selectivity = 1")
plt.xlabel("mean TI in motor + premotor (V/m)"); plt.ylabel("mean TI in right hippocampus (V/m)")
plt.legend(); plt.title("11,628 montages at 2 mA per pair"); plt.show()

**Reading the plot.** Each dot is one montage. Up means more field in the target; left means less in the regions to avoid. The most *intense* montage is rarely the most *selective* one, and choosing between them is a trade-off you have to make.

> **Try it:** (1) Add positions to `POOL` (for example `TP7`, `TP8`, `FT7`, `FT8`). How many montages do you get, and does the best one change? (2) Score by the 99th percentile instead of the mean. (3) Simulate the best leadfield montage with Step 6's code (normal FEM, 10 mm electrodes) and check that you get the same hippocampus value.

---
## Step 8 — Optimize the TI montage with `TesFlexOptimization` ⏳ (1 h or more per run)

The leadfield search in Step 7 was limited to cap positions and small electrodes. `TesFlexOptimization` drops both limits: it moves electrodes of any size **freely over the scalp**, guided by a *differential evolution* search, and keeps the best montage for a **goal**.

**Why it is fast enough.** It never re-meshes the head while searching:
* an electrode is modelled as the **skin nodes under its footprint**, with current injected there;
* moving an electrode then changes only the right-hand side **b** of the FEM system **A·v = b**, while the big matrix **A** (mesh + conductivities) stays the same;
* SimNIBS **factorizes A once** with a direct solver (PARDISO, or MUMPS on macOS). After that, each candidate position is a cheap solve that reads the field only in the ROIs;
* at the end, it runs normal simulations with properly meshed electrodes for the winning montage.

The stored factorization needs a **lot of RAM**. SimNIBS's documentation mentions ~12 GB for PARDISO. Run this on a machine with plenty of memory, or on a cluster.

| `goal` | Meaning |
|---|---|
| `"mean"` | maximize the mean TI in the ROI (**intensity**) |
| `"focality"` | high TI in the ROI **and** low TI in the non-ROI. `threshold = [non-ROI, ROI]` in V/m: TI should be below 0.24 V/m in the non-ROI and above 0.3 V/m in the ROI |

`e_postproc = "max_TI"` tells it to score the TI envelope from Step 6, not a single field.

Our settings: 2 pairs, circular electrodes with 10 mm radius, ±2 mA per pair; target = right hippocampus (charm label 53, GM); non-ROI (focality only) = motor + premotor cortex (Step 5E).

These runs are slow on a laptop. Set the `RUN_...` switches to `True` when you're ready, or run them on a cluster. When a run finishes, SimNIBS writes into its `output_folder`:
* `summary.txt`: settings, optimal electrode positions, and final field metrics;
* `final_sim_0/`, `final_sim_1/`: a normal tDCS simulation of each optimized pair, with real electrodes;
* `ernie_tes_flex_opt*.msh`: ready-made visualizations (opened automatically if `open_in_gmsh` is True).

### 8.1 Intensity: maximize the mean TI in the hippocampus

In [ ]:
RUN_OPT_INTENSITY = False   # ⏳

# the optimizer loads its own head mesh: free what we no longer need first
for name in ["ti_mesh", "magE1", "magE2", "ti"]:
    globals().pop(name, None)
gc.collect()

opt = opt_struct.TesFlexOptimization()
opt.subpath = str(M2M)
opt.output_folder = str(WORK_DIR / "opt_TI_intensity")
opt.open_in_gmsh = OPEN_GMSH

opt.goal = "mean"
opt.e_postproc = "max_TI"

for _ in range(2):                                   # two electrode pairs
    pair = opt.add_electrode_layout("ElectrodeArrayPair")
    pair.radius = [10]                               # mm
    pair.current = [0.002, -0.002]                   # A (2 mA)

roi = opt.add_roi()                                  # target: right hippocampus (GM)
roi.method = "volume"
roi.mask_path = str(fn_hippo_charm)
roi.mask_space = "subject"
roi.mask_value = 1

if RUN_OPT_INTENSITY:
    opt.run()
else:
    print("Set RUN_OPT_INTENSITY = True to run.")

### 8.2 Focality: hippocampus vs motor + premotor cortex

Same as above, plus a **second ROI**, which counts as the non-ROI when `goal = "focality"`, and the thresholds.

In [ ]:
RUN_OPT_FOCALITY = False   # ⏳

opt_f = opt_struct.TesFlexOptimization()
opt_f.subpath = str(M2M)
opt_f.output_folder = str(WORK_DIR / "opt_TI_focality")
opt_f.open_in_gmsh = OPEN_GMSH

opt_f.goal = "focality"
opt_f.threshold = [0.24, 0.3]          # V/m: [non-ROI, ROI]
opt_f.e_postproc = "max_TI"

for _ in range(2):
    pair = opt_f.add_electrode_layout("ElectrodeArrayPair")
    pair.radius = [10]
    pair.current = [0.002, -0.002]

roi = opt_f.add_roi()                  # 1st ROI = target
roi.method = "volume"
roi.mask_path = str(fn_hippo_charm)
roi.mask_space = "subject"
roi.mask_value = 1

non_roi = opt_f.add_roi()              # 2nd ROI = non-ROI (same definition as Step 5E)
non_roi.method = "volume_from_surface"
non_roi.surface_type = "central"
non_roi.mask_path = paths
non_roi.mask_space = spaces
non_roi.mask_value = values
non_roi.mask_operator = ["intersection"] + ["union"] * (len(paths) - 1)
non_roi.surface_inclusion_radius = 5

if RUN_OPT_FOCALITY:
    opt_f.run()
else:
    print("Set RUN_OPT_FOCALITY = True to run.")

### 8.3 Evaluate the optimized montages

We re-use exactly the tools from Step 6: the two `final_sim_*` folders are ordinary tDCS results, so `compute_ti` and `region_table` work unchanged. We also print the hand-picked montage from Step 6 for comparison.

Look beyond the numbers that were optimized. The focality run only penalised TI in the **motor/premotor** cortex. Open the optimized result in Gmsh (`ernie_tes_flex_opt*.msh` in the output folder) and look at TI on the grey-matter surface: does the field land in other cortical areas that nobody told the optimizer to avoid?

In [ ]:
def final_sim_files(folder):
    f1 = sorted(glob.glob(str(Path(folder) / "final_sim_0" / "*_scalar.msh")))
    f2 = sorted(glob.glob(str(Path(folder) / "final_sim_1" / "*_scalar.msh")))
    return (f1[0], f2[0]) if f1 and f2 else None

results = {"Step 6 (F5-P5 / F6-P6, 1 mA)": (fn_pair1, fn_pair2)}
for label, folder in [("opt intensity", WORK_DIR / "opt_TI_intensity"),
                      ("opt focality", WORK_DIR / "opt_TI_focality")]:
    files = final_sim_files(folder)
    if files:
        results[label] = files
    else:
        print(f"{label}: no results yet in {folder}")

tables = []
for label, (f1, f2) in results.items():
    m, e1, e2, t = compute_ti(f1, f2)
    tab = region_table(m, {"TI": t})
    tab.columns = [f"{c[0]} TI" for c in tab.columns]
    tab.insert(0, "montage", label)
    tables.append(tab.reset_index())
    del m, e1, e2, t; gc.collect()     # free memory before loading the next montage
pd.concat(tables).set_index(["montage", "region"])

In [ ]:
for folder in [WORK_DIR / "opt_TI_intensity", WORK_DIR / "opt_TI_focality"]:
    fn = folder / "summary.txt"
    if fn.exists():
        print("=" * 30, folder.name, "=" * 30)
        print(fn.read_text())

> **Try it:** (1) The Step 6 montage uses 1 mA and the optimized ones 2 mA per pair. Fields scale linearly with current, so halve the optimized TI values and compare fairly. (2) Change the thresholds to `[0.1, 0.2]` and re-run focality: how do the electrode positions change? (3) Use only the **right** motor/premotor cortex as the non-ROI (hint: keep only the `rh` entries in `paths`, `spaces` and `values` from Step 5E).

---
## Where to go next
* SimNIBS documentation and tutorials: <https://simnibs.github.io/simnibs>
* More example scripts ship with SimNIBS, in `simnibs/examples/` inside the installation (`simulations/TI.py`, `tes_flex_optimization/`, `optimization/TI_fields_leadfields.py`, `utilities/roi_definition.py`).
* **Leadfield approach:** for searching thousands of montages on a fixed EEG cap, it is much faster to simulate every electrode once (a *leadfield*) and then combine fields algebraically (`optimization/TI_fields_leadfields.py`). This is what the lab's pipeline does.